In [1]:
# # Delete all files in processed folder
# from notebookutils import mssparkutils  

# folder_path = "Files/processed/"
# files = mssparkutils.fs.ls(folder_path)

# for file in files:
#     print(f"Deleting file: {file.path}")
#     mssparkutils.fs.rm(file.path, False)



StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 3, Finished, Available, Finished, False)

Deleting file: abfss://392a9fab-a44d-4ccd-9ee0-0768735d515a@onelake.dfs.fabric.microsoft.com/3bdb84cb-797e-4cd5-ba43-93073debdcfd/Files/processed/dim_customer_2026-07-28.csv
Deleting file: abfss://392a9fab-a44d-4ccd-9ee0-0768735d515a@onelake.dfs.fabric.microsoft.com/3bdb84cb-797e-4cd5-ba43-93073debdcfd/Files/processed/dim_date_2026-07-28.csv
Deleting file: abfss://392a9fab-a44d-4ccd-9ee0-0768735d515a@onelake.dfs.fabric.microsoft.com/3bdb84cb-797e-4cd5-ba43-93073debdcfd/Files/processed/dim_promotion_2026-07-28.csv
Deleting file: abfss://392a9fab-a44d-4ccd-9ee0-0768735d515a@onelake.dfs.fabric.microsoft.com/3bdb84cb-797e-4cd5-ba43-93073debdcfd/Files/processed/dim_store_2026-07-28.csv
Deleting file: abfss://392a9fab-a44d-4ccd-9ee0-0768735d515a@onelake.dfs.fabric.microsoft.com/3bdb84cb-797e-4cd5-ba43-93073debdcfd/Files/processed/fullload_sales_asof_2026-07-14.csv
Deleting file: abfss://392a9fab-a44d-4ccd-9ee0-0768735d515a@onelake.dfs.fabric.microsoft.com/3bdb84cb-797e-4cd5-ba43-93073debdcfd

In [13]:
# Delete all tables
tables = spark.catalog.listTables()
display(tables)

for table in tables:
    table_name = table.name
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")


StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4c018531-9927-4edb-a548-34583046fc8b)

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from delta.tables import DeltaTable

try:
    fs = notebookutils.fs
except NameError:
    fs = mssparkutils.fs

raw_folder = "Files/raw"
audit_table_name = "load_control"


StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 6, Finished, Available, Finished, False)

In [14]:
spark.sql("""CREATE TABLE IF NOT EXISTS load_control (
    source_file STRING, file_date String, target_table String, loaded_at TIMESTAMP, rows_merged INT, rows_updated INT, rows_inserted INT
) USING DELTA""")

StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 17, Finished, Available, Finished, False)

DataFrame[]

In [15]:

DIM_STORE_SCHEMA = ["store_id", "store_code", "store_name", "city", "region", "store_type",
             "open_date", "square_feet", "employee_count"]

DIM_PROMOTION_SCHEMA = ["promotion_id", "promo_code", "promotion_name", "promo_type",
                  "discount_pct", "start_date", "end_date", "applies_to_category"]

DIM_DATE_SCHEMA = ["date_key", "date", "year", "quarter", "quarter_label", "month_number",
            "month_name", "month_short", "iso_week", "day_of_month", "day_of_week",
            "day_name", "is_weekend", "holiday_name", "is_holiday", "fiscal_year"]

DIM_CUSTOMER_SCHEMA = ["customer_id", "customer_code", "first_name", "last_name", "email",
                 "segment", "preferred_channel", "home_region", "join_date",
                 "age_band", "is_churned"]

SALES_COLS_SCHEMA = ["order_id","line_number","transaction_ts","store_id","customer_id","product_id",
              "quantity","unit_price","discount_amount","payment_method","order_status","source_system"]

files_schemas_tables = [
    ("dim_store",DIM_STORE_SCHEMA,"bronze_store"),
    ("dim_promotion",DIM_PROMOTION_SCHEMA,"bronze_promotion"),
    ("dim_date",DIM_DATE_SCHEMA,"bronze_date"),
    ("dim_customer",DIM_CUSTOMER_SCHEMA,"bronze_customer"),
    ("fullload_sales",SALES_COLS_SCHEMA,"bronze_sales")
]

brnz_slvr_tabels = { #skip the sales
    "bronze_store" : "silver_store",
    "bronze_promotion" : "silver_promotion",
    "bronze_date" : "silver_date",
    "bronze_customer" : "silver_customer"
}

product_file_name = "products.json"


StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 18, Finished, Available, Finished, False)

In [24]:
def isNotDuplicateFile(fname, date, table_name="load_control"):
    """
    Checks the audit table for an existing row matching this filename + date.
    Returns True if no such record exists yet (safe to process),
    False if it's already been logged (duplicate/idempotency guard).
    """
    # Table may not exist yet on the very first run
    if not spark.catalog.tableExists(table_name):
        return True

    match_count = (
        spark.table(table_name)
        .filter(
            (F.col("source_file") == fname) &
            (F.to_date(F.col("file_date")) == F.to_date(F.lit(date)))
        )
        .count()
    )
    return match_count == 0

def find_file(filename):
        hits = [f.name for f in fs.ls(raw_folder) if f.name.startswith(filename)]
        if len(hits) == 1:
            return hits[0]
        # elif len(hits) == 0:
        #     raise FileNotFoundError(f"'{filename}' not found in {raw_folder}")
        # else:
        #     raise ValueError(f"Multiple matches for '{filename}' in {raw_folder}: {hits}")

def write_csv_to_bronze(df, table_name):
    df_with_ts = df.withColumn("bronze_load_ts", F.current_timestamp())
    df_with_ts.write.format("delta").mode("append").saveAsTable(table_name)
    n = spark.table(table_name).count()
    print(f"{table_name} : {n:,} rows ")
    return n

def write_audit_record(src_filename: str, target_table: str, file_date, rows, **extra_cols):
    # Build the row as a dict so extra columns are easy to bolt on
    row = {"source_file": src_filename,"target_table": target_table, "file_date": file_date, **extra_cols} 
    df = spark.createDataFrame([tuple(row.values())], list(row.keys()))
    df = df.withColumn("rows_inserted", F.lit(rows))
    df = df.withColumn("loaded_at", F.current_timestamp())
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(audit_table_name)
    print(f"AUDIT: logged {src_filename} - {file_date}")

def readCSVFile(filename, schema_cols):
    schema = StructType([StructField(c, StringType(), True) for c in schema_cols])
    df = (spark.read
        .schema(schema)                              
        .option("header", True)
        .option("badRecordsPath", "Files/quarantine/badrecords")   # structural failures go here
        .csv(raw_folder + "/" + filename))
    display(filename)
    return df

def getFileDate(filename):
    fname, date_with_ext = filename.rsplit("_", 1)
    date = date_with_ext.replace(".csv", "")
    return date

def mvFileToProcessed(file):
    mssparkutils.fs.mv(
        f"{raw_folder}/{file}",
        f"Files/processed/{file}",
        True
    )

def clean_df(df):
    df = df.dropDuplicates()
    df = df.dropna(how = "all")
    return df

def write_delta(src, target, df):
    df.write.mode("overwrite").format("delta").saveAsTable(target)
    write_audit_record(src, target, "",df.count())


## Write Products to Silver
def load_products():
    product_file = find_file(product_file_name)
    if product_file:
        silver_products = (spark.read
            .option("multiline", "true")
            .json(raw_folder + "/" + product_file)
            .select(
                "product_id", "sku", "product_name", "category", "subcategory", "brand",
                F.col("unit_cost").cast("decimal(10,2)").alias("unit_cost"),
                F.col("list_price").cast("decimal(10,2)").alias("list_price"),
                "launch_date", "is_active",
                F.col("attributes.color").alias("attr_color"),
                F.col("attributes.weight_kg").alias("attr_weight_kg"),
                F.col("attributes.rating").alias("attr_rating")))
        silver_products.write.format("delta").mode("overwrite").saveAsTable("silver_products")
        write_audit_record(product_file_name, "silver_products", "",spark.table('silver_products').count())
        print(f"silver_products: {spark.table('silver_products').count()} rows")
        mvFileToProcessed(product_file)

def clean_sales(df_raw):
    """Bronze -> Silver transform for Meridian sales. Returns (valid_df, quarantine_df)."""
    typed = (df_raw
        .dropDuplicates(["order_id", "line_number"])                              # pattern 1
        .withColumn("transaction_ts",
            F.coalesce(F.to_timestamp("transaction_ts"),                          # pattern 2
                       F.to_timestamp("transaction_ts", "MM/dd/yyyy HH:mm")))
        .withColumn("store_id",    F.col("store_id").cast("int"))
        .withColumn("customer_id",
            F.coalesce(F.col("customer_id").cast("int"), F.lit(-1)))             # pattern 4b: guest -> -1
        .withColumn("product_id",  F.col("product_id").cast("int"))
        .withColumn("quantity",    F.col("quantity").cast("int"))
        .withColumn("unit_price",
            F.regexp_replace("unit_price", "[$]", "").cast("decimal(10,2)"))     # pattern 3
        .withColumn("discount_amount", F.col("discount_amount").cast("decimal(10,2)"))
        .withColumn("order_status", F.initcap("order_status")))                   # pattern 4a
    is_valid = ((F.col("quantity") > 0) &                                         # pattern 5
                F.col("transaction_ts").isNotNull() &
                F.col("unit_price").isNotNull())
    return typed.filter(is_valid), typed.filter(~is_valid)

def get_silver_watermark(target_table: str, control_table: str = "load_control"):
    """Returns the max bronze_load_ts already merged into silver for this source.
       Falls back to epoch start if nothing's been loaded yet."""
    if not spark.catalog.tableExists(control_table):
        print("No cntrl table@@@")
        return "1900-01-01 00:00:00"

    # Check if watermark_ts column exists (it's added on first incremental merge)
    df = spark.table(control_table)
    if "watermark_ts" not in df.columns:
        print("NO wtrmrk col########")
        return "1900-01-01 00:00:00"

    result = (df.filter(F.col("target_table") == target_table)
              .agg(F.max("watermark_ts").alias("wm"))
              .collect()[0]["wm"])
    return result if result is not None else "1900-01-01 01:00:00"

def promote_new_bronze_sales_to_silver():
    watermark = get_silver_watermark("bronze_sales")
    print(f"watermark: {watermark}")
    new_bronze = spark.table("bronze_sales").filter(F.col("bronze_load_ts") > watermark)
    #display(new_bronze)
    if new_bronze.isEmpty():
        print("no new bronze rows since last watermark")
        return

    batch, batch_quar = clean_sales(new_bronze)   
    batch = batch.cache()

    print(f"new bronze rows   : {new_bronze.count():,}")
    print(f"clean merge batch : {batch.count():,}")
    print(f"batch quarantined : {batch_quar.count():,}")

    silver_target = DeltaTable.forName(spark, "silver_sales")
    (silver_target.alias("t")
        .merge(batch.alias("s"), "t.order_id = s.order_id AND t.line_number = s.line_number")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

    # advance the watermark to the max bronze_load_ts just processed
    new_watermark = new_bronze.agg(F.max("bronze_load_ts")).collect()[0][0]
    spark.createDataFrame(
        [("bronze_sales", new_watermark, batch.count())],
        "target_table STRING, watermark_ts TIMESTAMP, rows_merged INT"
    ).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("load_control")
    print(f"watermark advanced to: {new_watermark}")    

StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 27, Finished, Available, Finished, False)

In [17]:
# Move all raw files to bronze table. Steps are:
# 1. read raw folder
# 2. loop thru all files in list, 
# 3.    for each file: pick file from folder, split name & timestamp
# 4.    write timestamp & file to audit table
# 5.    write content to bronze table
# 6.     mv file to processed

for fname, schema, tblname in files_schemas_tables: 
    file = find_file(fname)
    if file:
        df = readCSVFile(file, schema)
        date = getFileDate(file)
        if isNotDuplicateFile(file, date):
            write_csv_to_bronze(df,tblname)
            write_audit_record(fname,tblname, date, df.count())
            mvFileToProcessed(file)      

StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 20, Finished, Available, Finished, False)

'dim_store_2026-07-28.csv'

bronze_store : 21 rows 
AUDIT: logged dim_store - 2026-07-28


'dim_promotion_2026-07-28.csv'

bronze_promotion : 45 rows 
AUDIT: logged dim_promotion - 2026-07-28


...

bronze_date : 1,461 rows 
AUDIT: logged dim_date - 2026-07-28


'dim_customer_2026-07-28.csv'

bronze_customer : 1,200 rows 
AUDIT: logged dim_customer - 2026-07-28


'fullload_sales_asof_2026-07-14.csv'

bronze_sales : 361,892 rows 
AUDIT: logged fullload_sales - 2026-07-14


In [18]:
#load Products directly to silver for now
load_products()

StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 21, Finished, Available, Finished, False)

AUDIT: logged products.json - 
silver_products: 250 rows


In [25]:
# Move initial load from all Bronze tables(except sales) to Silver after generic cleaning
for brnz_tbl, silver_tbl in brnz_slvr_tabels.items():
    df = spark.table(brnz_tbl) 
    print(f"Cleaning...: {brnz_tbl} - {df.count()}")
    df = clean_df(df)
    print(f"Cleaned: Records - {df.count()}")
    write_delta(brnz_tbl, silver_tbl,df)

StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 28, Finished, Available, Finished, False)

Cleaning...: bronze_store - 21
Cleaned: Records - 21
AUDIT: logged bronze_store - 
Cleaning...: bronze_promotion - 45
Cleaned: Records - 45
AUDIT: logged bronze_promotion - 
Cleaning...: bronze_date - 1461
Cleaned: Records - 1461
AUDIT: logged bronze_date - 
Cleaning...: bronze_customer - 1200
Cleaned: Records - 1200
AUDIT: logged bronze_customer - 


In [26]:
## Load from bronze to Silver Sales initial Full Load 
silver_df, quarantine_df = clean_sales(spark.table("bronze_sales"))
write_delta("bronze_sales","silver_sales",silver_df)
write_delta("bronze_sales","quarantine_sales",quarantine_df)

print(f"silver_sales     : {spark.table('silver_sales').count():,} ")
print(f"quarantine_sales : {spark.table('quarantine_sales').count():,} ")
print(f"guest lines (-1) : {spark.table('silver_sales').filter('customer_id = -1').count():,} ")

StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 29, Finished, Available, Finished, False)

AUDIT: logged bronze_sales - 
AUDIT: logged bronze_sales - 
silver_sales     : 356,182 
quarantine_sales : 2,860 
guest lines (-1) : 43,323 


In [ ]:
### Append Incremental Loads ####

In [30]:
## Bronze Sales Incremental append
#raw_batch = (spark.read.schema(sales_schema).option("header", True).csv(incremental_file_path))

incremental_sales_file = "incremental_sales"

incre_sales_dated_file = find_file(incremental_sales_file)
if incre_sales_dated_file:
    raw_batch_df = readCSVFile(incre_sales_dated_file, SALES_COLS_SCHEMA)
    date = getFileDate(incre_sales_dated_file)
    if isNotDuplicateFile(incre_sales_dated_file, date):
        write_csv_to_bronze(raw_batch_df,"bronze_sales")
        write_audit_record(incremental_sales_file, "bronze_sales", date,raw_batch_df.count())
        mvFileToProcessed(incre_sales_dated_file)

print("Incremental sales processed")

StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 33, Finished, Available, Finished, False)

'incremental_sales_week2_2026-07-28.csv'

bronze_sales : 369,796 rows 
AUDIT: logged incremental_sales - 2026-07-28
Incremental sales processed


In [31]:
## Silver Sales Incremental merge
promote_new_bronze_sales_to_silver()


StatementMeta(, fbcec6e0-9832-48d7-9b9a-985142a9ce29, 34, Finished, Available, Finished, False)

watermark: 2026-08-24 23:58:10.094431
new bronze rows   : 2,590
clean merge batch : 2,537
batch quarantined : 15
watermark advanced to: 2026-08-25 00:14:24.797408
